# 12 分箱模块 (`core.binning`)

演示数据：`hengshucredit_yyp.xlsx` 或 `hscredit_yyp.xlsx`（置于 `examples/`）。
目标：`MOB1 > 3` 为坏（与 `01_binning.ipynb` 一致）。

In [ ]:
import os, sys
from pathlib import Path

_project = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _project.replace("\\", "/") not in [p.replace("\\", "/") for p in sys.path]:
    sys.path.insert(0, _project)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from hscredit import init_setting
init_setting()


def load_demo_excel():
    roots = [Path.cwd(), Path.cwd() / "examples", Path.cwd().parent]
    for root in roots:
        for name in ("hengshucredit_yyp.xlsx", "hscredit_yyp.xlsx"):
            p = root / name
            if p.is_file():
                return pd.read_excel(p), p
    raise FileNotFoundError(
        "未找到演示数据：请将 hengshucredit_yyp.xlsx 或 hscredit_yyp.xlsx 放在 examples/ 目录。"
    )


df, DATA_PATH = load_demo_excel()
print("数据文件:", DATA_PATH, "形状:", df.shape)

if "MOB1" in df.columns:
    df = df.copy()
    df["target_demo"] = (pd.to_numeric(df["MOB1"], errors="coerce").fillna(-1) > 3).astype(int)
    target = "target_demo"
elif "FPD" in df.columns:
    df = df.copy()
    t = pd.to_numeric(df["FPD"], errors="coerce")
    df = df.loc[t.notna()].copy()
    df["target_demo"] = (t.loc[df.index] > 0).astype(int)
    target = "target_demo"
else:
    raise ValueError("需要 MOB1 或 FPD 列以构造二分类标签")

exclude = {"MOB1", "MOB2", "target_demo", "CURRENT_DPD"}
num_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
print("目标:", target, "坏样本率:", f"{df[target].mean():.2%}", "数值列数:", len(num_cols))


## 1. `OptimalBinning` 统一接口

In [ ]:
from hscredit import OptimalBinning, bin_plot
import matplotlib.pyplot as plt

def pick_numeric_feature():
    for c in num_cols:
        if 'C3' in str(c):
            return c
    return num_cols[0] if num_cols else df.columns[0]

feat = pick_numeric_feature()
X = df[[feat]].copy()
y = df[target]

for method in ('mdlp', 'best_iv', 'best_ks'):
    b = OptimalBinning(method=method, max_n_bins=5, min_n_bins=2, verbose=False)
    b.fit(X, y)
    print(method, 'bins:', b.n_bins_, 'splits:', b.splits_.get(feat))

try:
    b_or = OptimalBinning(method='or_tools', max_n_bins=5, min_n_bins=2, verbose=False)
    b_or.fit(X, y)
    print('or_tools bins:', b_or.n_bins_)
    print(b_or.get_bin_table(feat).head(8).to_string())
except Exception as e:
    print('or_tools 跳过:', e)

b_mdlp = OptimalBinning(method='mdlp', max_n_bins=5, verbose=False)
b_mdlp.fit(X, y)
bin_plot(b_mdlp.get_bin_table(feat), title=f'{feat} MDLP')
plt.show()


## 2. 直接实例化分箱类

In [ ]:
from hscredit import QuantileBinning

qb = QuantileBinning(max_n_bins=5)
qb.fit(df[[feat]], y)
print(qb.get_bin_table(feat)[['分箱标签', '样本占比', '坏样本率', 'LIFT值']].head())
